### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
# Initilaize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

# Step 1: Split the text into sentences
sentences = text.strip().split('\n')

# Step 2: Generate embeddings for each sentence
embeddings = model.encode(sentences)

# Step 3: Initialize parameters
threshold = 0.7 # control the chunk tightness
chunks = []
current_chunk = [sentences[0]]

# Step 4: Semantically group sentences based on threshold
for i in range(1, len(sentences)):
    # Calculate cosine similarity between the current sentence and the last sentence in the current chunk
    sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
    
    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(' '.join(current_chunk))
        current_chunk = [sentences[i]]
    
# Add the last chunk
chunks.append(' '.join(current_chunk))

# Output the chunks
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}:\n{chunk}\n")

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.



### RAG Pipeline Modular Coding

In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [16]:
### Custom Semantic Chunking with Threshold
class ThresholdSemanticChunker():
    def __init__(self, model_name='all-MiniLM-L6-v2', threshold=0.7):
        self.threshold = threshold
        self.model = SentenceTransformer(model_name)

    def split(self, text):
        sentences = text.strip().split('\n')
        embeddings = self.model.encode(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i-1], embeddings[i]])[0][1]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentences[i]]

        if current_chunk:
            chunks.append(' '.join(current_chunk))

        return chunks
    
    def split_documents(self, documents):
        result = []
        for doc in documents:
            chunks = self.split(doc.page_content)
            for chunk in chunks:
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result
    
    

In [17]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [18]:
### Chunking
chunker = ThresholdSemanticChunker(threshold=0.7)
chunks = chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [20]:
### Vector Store

import os
os.environ["GITHUB_API_KEY"]=os.getenv("GITHUB_API_KEY")

embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key = os.getenv("GITHUB_API_KEY"),
    openai_api_base = "https://models.inference.ai.azure.com",
    dimensions= 1536
)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever()



In [21]:
## Prompt Template
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [22]:
## LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0.7,
    api_key=os.getenv("GROQ_API_KEY")
)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000120CF65D8E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000120C78B98E0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [23]:
## LCEL chain with retriever and prompt

rag_chain = (
    RunnableMap(
        {
            "context": lambda inputs: retriever.invoke(inputs["question"]),
            "question": lambda inputs: inputs["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# Run query
query = {"question": "What is langchain used for?"}
answer = rag_chain.invoke(query)
print(answer)

According to the provided context, LangChain is a framework for building applications with Large Language Models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


#### Semantic Chunker with Langchain

In [24]:
from  langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader

In [28]:
## Load the documents
loader = TextLoader('langchain_intro.txt')
docs = loader.load()


# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key = os.getenv("GITHUB_API_KEY"),
    openai_api_base = "https://models.inference.ai.azure.com",
    dimensions= 1536
)

## Create the semantic chunker
chunker = SemanticChunker(embeddings=embeddings)

## Split the document into semantic chunks
chunks = chunker.split_documents(docs)

for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}:\n{chunk.page_content}\n")

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

Chunk 2:
The Eiffel Tower is located in Paris. France is a popular tourist destination.



In [30]:
from langchain_experimental.text_splitter import SemanticChunker

## Create the threshold semantic chunker
threshold_chunker = SemanticChunker(embeddings=embeddings,
                                      breakpoint_threshold_amount=0.7)

## Split the document into threshold semantic chunks
threshold_chunks = threshold_chunker.split_documents(docs)

for idx, chunk in enumerate(threshold_chunks):
    print(f"Threshold Chunk {idx+1}:\n{chunk.page_content}\n")

Threshold Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Threshold Chunk 2:
You can create chains, agents, memory, and retrievers.

Threshold Chunk 3:
The Eiffel Tower is located in Paris.

Threshold Chunk 4:
France is a popular tourist destination.



In [32]:
vector_store = FAISS.from_documents(threshold_chunks, embeddings)
retriever = vector_store.as_retriever()

#### LCEL implementation

In [33]:
## Prompt Template
template = """Answer the question based on the following context:

{context}

Question: {question}
"""


prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [34]:
rag_chain = (
    RunnableMap(
        {
            "context": lambda inputs: retriever.invoke(inputs["question"]),
            "question": lambda inputs: inputs["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

query = {"question": "What is langchain used for?"}
answer = rag_chain.invoke(query)
print(answer)

Based on the provided context, LangChain is used for building applications with Large Language Models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
